In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

dfs = [pd.read_csv("C:\\Users\\reach\\OneDrive\\Code files\\SEM4\\MFC4\\realdata\\BCICIV_calib_ds1" + user + ".csv") for user in ['a', 'b', 'c', 'd']]


for i in range(len(dfs)):
    dfs[i]['User'] = pd.Series(i, index = dfs[i].index)

data = pd.concat(dfs, axis=0).sample(frac=1.0, random_state = 123).reset_index(drop=True)

data = data.drop(['User'], axis=1)
print(data)



     Class  AF3 delta std   AF3 delta m  AF3 theta std   AF3 theta m  \
0        2   4.459960e+06  3.770835e+06   5.389005e+05  4.599216e+05   
1        2   1.128200e+07  9.704838e+06   3.682426e+06  3.371920e+06   
2        2   8.709910e+03  6.943142e+03   5.001467e+03  4.813162e+03   
3        1   1.813854e+04  2.118412e+04   7.081516e+03  5.504063e+03   
4        2   3.359556e+04  2.466924e+04   1.241038e+04  1.309774e+04   
..     ...            ...           ...            ...           ...   
795      2   7.115694e+04  5.918708e+04   2.833706e+04  2.835565e+04   
796      1   2.658777e+04  2.230288e+04   9.138203e+03  7.517715e+03   
797      1   2.289223e+04  1.732716e+04   4.521185e+03  4.464849e+03   
798      2   2.083043e+04  1.006488e+04   4.971226e+03  5.103167e+03   
799      1   7.379608e+06  7.240081e+06   2.641389e+06  2.441139e+06   

     AF3 alpha std    AF3 alpha m  AF3 beta std     AF3 beta m  AF4 delta std  \
0       66164.7800   55179.732751    6182.25600    522

In [12]:
# Assuming data is already defined in the previous cell
X = data.drop(columns=['Class']).values
y = data['Class'].values
y-=1

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Reshape X_tensor to have the correct shape for Conv1d
X_tensor = X_tensor.unsqueeze(1)  # Add a channel dimension

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

class MDCBN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(MDCBN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=input_channels, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        self.lstm = nn.LSTM(128, 64, batch_first=True, bidirectional=True)

        self.fc1 = nn.Linear(64 * 2, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))

        x = x.permute(0, 2, 1)  # Prepare for LSTM
        x, _ = self.lstm(x)
        x = x[:, -1, :]

        x = self.relu(self.bn3(self.fc1(x)))
        x = self.fc2(x)
        return x

# Model setup
input_size = X_tensor.shape[1]
num_classes = 2
model = MDCBN(input_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 100
for epoch in range(epochs):
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Save trained model
torch.save(model.state_dict(), "mdcbn_modelv2.pth")
print("Model training complete and saved as mdcbn_model.pth")

# Calculate accuracy
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in dataloader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs.data, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = correct / total
print(f"Model Accuracy: {accuracy:.4f}")


Epoch 1/100, Loss: 0.7438
Epoch 2/100, Loss: 0.6973
Epoch 3/100, Loss: 0.6978
Epoch 4/100, Loss: 0.6737
Epoch 5/100, Loss: 0.6840
Epoch 6/100, Loss: 0.6726
Epoch 7/100, Loss: 0.7172
Epoch 8/100, Loss: 0.6282
Epoch 9/100, Loss: 0.6762
Epoch 10/100, Loss: 0.7376
Epoch 11/100, Loss: 0.7164
Epoch 12/100, Loss: 0.6155
Epoch 13/100, Loss: 0.6771
Epoch 14/100, Loss: 0.7052
Epoch 15/100, Loss: 0.5863
Epoch 16/100, Loss: 0.6829
Epoch 17/100, Loss: 0.6795
Epoch 18/100, Loss: 0.6764
Epoch 19/100, Loss: 0.6901
Epoch 20/100, Loss: 0.6540
Epoch 21/100, Loss: 0.6186
Epoch 22/100, Loss: 0.7258
Epoch 23/100, Loss: 0.6543
Epoch 24/100, Loss: 0.6261
Epoch 25/100, Loss: 0.7516
Epoch 26/100, Loss: 0.5689
Epoch 27/100, Loss: 0.6354
Epoch 28/100, Loss: 0.5328
Epoch 29/100, Loss: 0.5989
Epoch 30/100, Loss: 0.5366
Epoch 31/100, Loss: 0.6272
Epoch 32/100, Loss: 0.5784
Epoch 33/100, Loss: 0.6028
Epoch 34/100, Loss: 0.7267
Epoch 35/100, Loss: 0.6892
Epoch 36/100, Loss: 0.5313
Epoch 37/100, Loss: 0.6669
Epoch 38/1